# Support Vector Machine (SVM) Model

Train and evaluate a Support Vector Machine classifier, run hyperparameter tuning with GridSearchCV, and plot the confusion matrix.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
import os
import numpy as np
import joblib
from pathlib import Path

DATA_DIR = Path("./processed_data")
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

if not npz_file.exists():
    print("Feature file not found. Automatically triggering feature extraction pipeline...")
    import sys
    sys.path.append(str(Path(".").resolve()))
    from src.feature_extraction import build_feature_matrices
    build_feature_matrices()

data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape}")
print(f"  Validation set: {X_val.shape}")
print(f"  Testing set   : {X_test.shape}")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

## 1. Train Baseline SVM Classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("--- Training Baseline SVM (RBF kernel) ---")
svm_baseline = SVC(kernel='rbf', random_state=42)
svm_baseline.fit(X_train, y_train)

y_pred_base = svm_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_base)
print(f"Baseline SVM Test Accuracy: {baseline_acc * 100:.2f}%")

## 2. Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

print("--- Hyperparameter Tuning with GridSearchCV ---")
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

grid_search_svm = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters : {grid_search_svm.best_params_}")
print(f"Best Cross-Val Score : {grid_search_svm.best_score_ * 100:.2f}%")

## 3. Evaluate & Save Best SVM Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

best_svm = grid_search_svm.best_estimator_
y_pred = best_svm.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
print(f"Final Tuned SVM Test Accuracy: {final_acc * 100:.2f}%")

# Save model
MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "svm_model.pkl"
joblib.dump(best_svm, model_path)
print(f"Best SVM model saved to '{model_path}' successfully!")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Tuned SVM')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()